# A 24-hour GPU Ocean / Norkyst twin experiment

Initialize GPU Ocean once from Norkyst, prescribe Norkyst boundaries, and verify the freely evolving interior against Norkyst sea level and **depth-averaged** currents. This measures one-way parent-model fidelity, not independent forecast accuracy.

Use the same shared implementation as `validate_norkyst.py` and the LUMI Slurm launcher. Run integration only in an allocated GPU environment, using the project's GPU-enabled kernel (on LUMI, `GPUOcean LUMI (uv)`). Saved results can be analysed using an ordinary CPU kernel. No simulation results are bundled or fabricated.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import csv
import numpy as np
from matplotlib import pyplot as plt
from gpuocean.utils.norkyst_twin import (
    TwinConfig, TwinExperiment, prepare, load_results, plot_results,
)

RUN_SIMULATION = False  # False: analyse an existing notebook or Slurm run.
RESULTS_DIR = Path('/path/to/norkyst-twin-results')
SOURCES = (
    '/path/to/NorKyst-800m_ZDEPTHS_his.an.2019071600.nc',
    '/path/to/compatible-continuation.nc',
)
config = TwinConfig(
    sources=SOURCES, output=str(RESULTS_DIR), start='2019-07-16T00:00:00Z',
    bounds=(1400, 1900, 450, 750), duration=24 * 3600,
    output_interval=3600, diagnostic_interval=600,
    sponge_cells=20, buffer_cells=10, courant_number=0.8,
    wind=True, pressure_variable=None, budget=False,
)
asdict(config)

## Review the configuration

Edit the paths before running. A 24-hour experiment needs 25 hourly records, including the next day's endpoint. Continuations must belong to a compatible integration; conflicting duplicate timestamps are rejected. Components must be collocated and expressed in the source grid axes; do not substitute surface currents or assume an unverified rotation.

`wind=True` requires real `Uwind`/`Vwind` fields. If they are unavailable, explicitly select `wind=False` and interpret the omission as a model mismatch. Set `pressure_variable` when compatible atmospheric pressure is available. Freshwater forcing is not applied. The initializer's friction coefficient is not assumed to match Norkyst's closure.

Set `budget=True` for expensive discrete RK2 accounting, not performance measurements. For the reduced-CFL sensitivity use a separate output directory and `courant_number=0.4`. For the enlarged-domain test, preserve the same source-index `verification_bounds`, safely outside both runs' sponge/buffer regions.

In [ ]:
prepared = prepare(config) if RUN_SIMULATION else None
if prepared is not None:
    print('Source records:', len(prepared.times))
    print('Elapsed source hours:', (prepared.times - prepared.times[0]) / 3600)
    print('Forcing assumptions:', prepared.forcing)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for ax, values, title in zip(axes, (prepared.source_depth, prepared.reference[0, 0]),
                                  ('Source bathymetry (m)', 'Initial Norkyst sea level (m)')):
        image = ax.imshow(np.where(prepared.wet, values, np.nan), origin='lower')
        ax.set_title(title)
        fig.colorbar(image, ax=ax)
    plt.show()
else:
    print('Analysis-only mode: no Norkyst input access or GPU initialization.')

## Interpret the input checks

Confirm continuous coverage through hour 24 and inspect the coastline and initial sea level. The preflight rejects gaps, missing wet-cell values, changing grids and unsupported units. The initial Norkyst sea-level mask defines the source wet region; later mask changes are rejected. Distinguish the raw source bathymetry from GPU Ocean's reconstructed bathymetry when comparing transports.

In [ ]:
if RUN_SIMULATION:
    with TwinExperiment(config, prepared=prepared) as experiment:
        for hour in range(1, 25):
            experiment.advance_to(hour * 3600)
            print(f'Completed {experiment.time / 3600:.2f} simulation hours')
            # Call plot_results(RESULTS_DIR, elapsed=hour * 3600) here for live plots.
else:
    print('Using saved results:', RESULTS_DIR)

## Integration status

The runner records hour zero and each hourly endpoint, while writing lightweight diagnostics every ten minutes. The context manager closes files on interruption and marks unfinished runs `incomplete`; do not present a partial run as a 24-hour validation. To pause and continue interactively inside the context, advance one hour at a time; exiting the context ends that run (restart from disk is not implemented). No warmup/reset or interior assimilation is performed.

In [ ]:
metadata, fields = load_results(RESULTS_DIR)
print('Run status:', metadata['status'])
print('Saved simulation hours:', fields['elapsed_seconds'] / 3600)
with (RESULTS_DIR / 'metrics.csv').open() as file:
    metrics = list(csv.DictReader(file))
interior = [r for r in metrics if r['region'] == 'interior' and float(r['time_s']) > 0]
for row in interior:
    if any(abs(float(row['time_s']) - h * 3600) < 1e-6 for h in (6, 12, 18, 24)):
        print({key: row[key] for key in ('time_s', 'eta_rmse_m', 'velocity_rmse_m_s',
                                        'eta_skill', 'velocity_skill')})
figures = plot_results(RESULTS_DIR, save=True)
plt.show()

## Interpret the model comparison

The difference maps now show **GPU Ocean minus Norkyst at the same time**. The initial-versus-later Norkyst difference is the persistence baseline, not the model error. Positive skill means improvement over persistence; skill is undefined when persistence error is zero (including initialization).

Use the fixed interior mask for the primary conclusion, and inspect boundary, buffer, shallow and offshore scores separately in `metrics.csv`. Boundary agreement is partly prescribed, and boundary information can propagate throughout the domain. The plotted interior sites are reproducible representative grid locations, not observations. One day is insufficient for robust tidal constituent fitting.

In [ ]:
with (RESULTS_DIR / 'diagnostics.csv').open() as file:
    diagnostics = list(csv.DictReader(file))
hours = np.array([float(r['time_s']) / 3600 for r in diagnostics])
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, name, unit in zip(axes, ('volume', 'momentum_x', 'momentum_y'), ('m3', 'm4/s', 'm4/s')):
    ax.plot(hours, [float(r[f'{name}_change']) for r in diagnostics], color='tab:blue', label='Integral change')
    if f'{name}_budget_residual' in diagnostics[0]:
        ax.plot(hours, [float(r[f'{name}_budget_residual']) for r in diagnostics],
                color='tab:red', label='Discrete budget residual')
    ax.set(xlabel='Elapsed hours', ylabel=unit, title=name)
    ax.legend()
plt.show()

## Interpret the budgets

Neither total mass nor momentum is expected to remain constant in this forced, open-boundary case. Without `budget=True`, these plots show integral changes only, not conservation errors. With accounting enabled, the residual subtracts accumulated numerical flux divergence, combined applied sources, friction, clipping and boundary-relaxation increments using RK2 stage weights. Physical source terms are currently grouped, not separately attributed to wind/Coriolis/pressure/topography. Internal MPI interfaces cancel through signed divergence, and halos are excluded. Remaining residuals include float32 update roundoff.

Momentum is density-free here; multiply by 1025 kg/m3 for physical momentum. Central-section transport diagnostics are cell-centred sampled approximations, not the solver's conservative face fluxes. Choose meaningful exchange/anomaly scales when normalizing residuals; resting-water volume alone can conceal errors.

### Q&A
After execution, state whether the 24-hour interior simulation improves on persistence for sea level and depth-averaged velocity, quoting the verified metrics above. No accuracy conclusion has been established in this unexecuted notebook.

### Data Analysis Key Findings
- Record the 6-, 12-, 18- and 24-hour scores and the coastal/offshore contrast.
- Distinguish initial mapping errors, model discrepancies, integral changes and measured budget residuals.

### Insights or Next Steps
- Repeat with reduced CFL, an enlarged domain over a fixed verification region, and multiple MPI ranks.
- Repeat the protocol for contrasting forcing/tidal conditions before making general validation claims.